In [ ]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', "1")

In [ ]:
# Root filesystem /tmp is full; keep temporary files on the larger /data volume.
import os
os.makedirs("/data/susanket/tmp", exist_ok=True)
os.environ["TMPDIR"] = "/data/susanket/tmp"

import io
import sys
import tqdm
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision import transforms
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

# Add your project src directory
sys.path.append(r"/home/susanket/esri-vlm/src")
from model import CLIP, build_model
from main import CLIPLightningModule

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
CKPT_PATH = "/home/susanket/esri-vlm/esri-vlm/ddvi5ics/checkpoints/epoch=51-step=307164.ckpt"  # Update path

ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)

if "hyper_parameters" in ckpt:
    # Clean up non-serializable or unnecessary hyperparameters
    for hp_key in ["eval_downstream", "air_temp_data_path", "election_data_path", "_instantiator"]:
        ckpt["hyper_parameters"].pop(hp_key, None)

    model_hp = dict(ckpt["hyper_parameters"])
    model_hp["load_text_backbone"] = True  # eval encodes raw text queries with Qwen
    lightning_model = CLIPLightningModule(**model_hp).to(device)
    lightning_model.load_state_dict(ckpt["state_dict"])
    lightning_model.eval()
    model = lightning_model.model
    tokenizer = lightning_model.tokenizer
else:
    model = build_model(ckpt["state_dict"]).to(device)
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B", padding_side="left")

model = model.to(device)
model.eval()

# Handle DINOv2 backbone if feature extraction is required
dino_backbone = None
if getattr(model.visual, "precomputed", False):
    from model import DINOV2_REPO
    try:
        dino_backbone = torch.hub.load(DINOV2_REPO, model.visual.model_name, trust_repo=True)
    except TypeError:
        dino_backbone = torch.hub.load(DINOV2_REPO, model.visual.model_name)
    dino_backbone.eval().to(device)

print("Model successfully loaded!")

In [ ]:
PARQUET_PATH = "/data/susanket/vlm/images3/image-00.parquet"  # Update path
IMAGE_COLUMN = "image_bytes"  # Column name containing raw image bytes or dict with 'bytes'

df = pd.read_parquet(PARQUET_PATH).sample(10_000)
print(f"Loaded {len(df)} samples from {PARQUET_PATH}")

image_transform = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def decode_and_transform(raw_image_data):
    """Helper to handle bytes directly or HuggingFace Datasets dict format."""
    if isinstance(raw_image_data, dict) and "bytes" in raw_image_data:
        raw_bytes = raw_image_data["bytes"]
    else:
        raw_bytes = raw_image_data

    try:
        img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
    except Exception:
        img = Image.new("RGB", (224, 224))
    return img

raw_pil_images = [decode_and_transform(row[IMAGE_COLUMN]) for _, row in tqdm.tqdm(df.iterrows())]
tensor_images = torch.stack([image_transform(img) for img in tqdm.tqdm(raw_pil_images)])

In [ ]:
BATCH_SIZE = 256
all_img_embeddings = []

with torch.no_grad():
    for i in range(0, len(tensor_images), BATCH_SIZE):
        batch = tensor_images[i : i + BATCH_SIZE].to(device)
        
        if dino_backbone is not None:
            feats = dino_backbone.forward_features(batch)
            batch = torch.cat([
                feats["x_norm_clstoken"].unsqueeze(1),
                feats["x_norm_patchtokens"]
            ], dim=1)

        embeddings = model.encode_image(batch)
        embeddings = F.normalize(embeddings, dim=-1)
        all_img_embeddings.append(embeddings.cpu().numpy())

image_embeddings = np.concatenate(all_img_embeddings, axis=0)
print(f"Computed embeddings matrix shape: {image_embeddings.shape}")

### Retreive

In [ ]:
QUERIES = [
    "residential buildings",
    "solar panel farm",
    "Solar Panels",
    "Red buildings by a river",
    "Edificios rojos junto a un río",
    "नदी के किनारे लाल इमारतें",
    "河边的红色建筑",
    "commercial airport runway",
    "dense forest and river",
    "bseball fields",
    "baseball fields",
    "sports fields"
]

with torch.no_grad():
    enc = tokenizer(
        QUERIES,
        padding=True,
        truncation=True,
        max_length=model.transformer.context_length,
        return_tensors="pt",
    )
    tokens = {k: v.to(device) for k, v in enc.items() if k in ("input_ids", "attention_mask")}
    
    text_embeddings = model.encode_text(tokens)
    text_embeddings = F.normalize(text_embeddings, dim=-1).cpu().numpy()

# Cosine similarity matrix: (num_queries, num_images)
similarity_matrix = text_embeddings @ image_embeddings.T

In [ ]:
TOP_K = 5

for q_idx, query in enumerate(QUERIES):
    scores = similarity_matrix[q_idx]
    top_k_indices = np.argsort(-scores)[:TOP_K]

    fig, axes = plt.subplots(1, TOP_K, figsize=(18, 4))
    fig.suptitle(f'Query: "{query}"', fontsize=14, fontweight="bold", y=1.05)

    for idx, img_idx in enumerate(top_k_indices):
        score = scores[img_idx]
        ax = axes[idx] if TOP_K > 1 else axes
        
        ax.imshow(raw_pil_images[img_idx])
        ax.set_title(f"Rank {idx+1}\nScore: {score:.4f}", fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import math

def plot_topk(query, topk = 100):
    with torch.no_grad():
        enc = tokenizer(
            [query],
            padding=True,
            truncation=True,
            max_length=model.transformer.context_length,
            return_tensors="pt",
        )
        tokens = {k: v.to(device) for k, v in enc.items() if k in ("input_ids", "attention_mask")}
        query_emb = model.encode_text(tokens)
        query_emb = F.normalize(query_emb, dim=-1).cpu().numpy()

    # 3. Compute cosine similarity across ALL dataset images
    all_scores = (query_emb @ image_embeddings.T).squeeze(0)  # Shape: (num_images,)

    # 4. Get indices of top TOP_K highest scores
    top_indices = np.argsort(-all_scores)[:topk]
    top_scores = all_scores[top_indices]

    # 5. Plot 10x10 top-100 collage
    fig, axes = plt.subplots(10, 10, figsize=(20, 2 * 10))
    fig.suptitle(f'Top {topk} Retrieved Images for Query:\n"{query}"', fontsize=16, fontweight="bold", y=1.02)

    axes = axes.flatten()

    for i in range(len(axes)):
        ax = axes[i]
        if i < topk:
            img_idx = top_indices[i]
            score = top_scores[i]
            
            ax.imshow(raw_pil_images[img_idx])
            ax.set_title(f"#{i+1} | {score:.3f}", fontsize=8, pad=3)
            ax.axis("off")
        else:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_topk("solar panel", topk = 20)

In [ ]:
plot_topk("swimming pool", topk = 50)

In [ ]:
plot_topk("edificio rojo, río", topk = 100)

---

### Patch Heatmap

In [ ]:
# ==========================================
# Setup & Environment (from your snippet)
# ==========================================
import os
os.makedirs("/data/susanket/tmp", exist_ok=True)
os.environ["TMPDIR"] = "/data/susanket/tmp"

import io
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision import transforms
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

# Add your project src directory
sys.path.append(r"/home/susanket/esri-vlm/src")
from model import CLIP, build_model
from main import CLIPLightningModule

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

CKPT_PATH = "/home/susanket/esri-vlm/esri-vlm/ddvi5ics/checkpoints/epoch=51-step=307164.ckpt"
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)

if "hyper_parameters" in ckpt:
    for hp_key in ["eval_downstream", "air_temp_data_path", "election_data_path", "_instantiator"]:
        ckpt["hyper_parameters"].pop(hp_key, None)

    model_hp = dict(ckpt["hyper_parameters"])
    model_hp["load_text_backbone"] = True  
    lightning_model = CLIPLightningModule(**model_hp).to(device)
    lightning_model.load_state_dict(ckpt["state_dict"])
    lightning_model.eval()
    model = lightning_model.model
    tokenizer = lightning_model.tokenizer
else:
    model = build_model(ckpt["state_dict"]).to(device)
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B", padding_side="left")

model = model.to(device)
model.eval()

# Handle DINOv2 backbone
dino_backbone = None
if getattr(model.visual, "precomputed", False):
    from model import DINOV2_REPO
    try:
        dino_backbone = torch.hub.load(DINOV2_REPO, model.visual.model_name, trust_repo=True)
    except TypeError:
        dino_backbone = torch.hub.load(DINOV2_REPO, model.visual.model_name)
    dino_backbone.eval().to(device)

print("Model successfully loaded!")

# ==========================================
# Visualization Function for 5 Phrases
# ==========================================
image_transform = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def decode_image(raw_image_data):
    """Helper to handle bytes directly or HuggingFace Datasets dict format."""
    if isinstance(raw_image_data, dict) and "bytes" in raw_image_data:
        raw_bytes = raw_image_data["bytes"]
    else:
        raw_bytes = raw_image_data

    try:
        img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
    except Exception:
        img = Image.new("RGB", (224, 224))
    return img

def visualize_phrase_heatmaps(parquet_row, queries, model, tokenizer, dino_backbone, image_column="image_bytes"):
    """
    Takes a single parquet row and 5 text phrases, computes patch-level similarity,
    and plots the original image alongside 5 heatmap overlays.
    """
    assert len(queries) == 5, "Please provide exactly 5 phrases!"
    
    # 1. Process Image
    raw_img = decode_image(parquet_row[image_column])
    pil_resized = raw_img.resize((224, 224), Image.Resampling.BICUBIC)
    
    img_tensor = image_transform(raw_img).unsqueeze(0).to(device) # [1, 3, 224, 224]

    with torch.no_grad():
        batch = img_tensor
        
        # Extract DINOv2 features
        if dino_backbone is not None:
            feats = dino_backbone.forward_features(batch)
            patch_tokens = feats["x_norm_patchtokens"] # Shape: [1, 256, dino_dim]
            cls_token = feats["x_norm_clstoken"].unsqueeze(1) # Shape: [1, 1, dino_dim]
            tokens = torch.cat([cls_token, patch_tokens], dim=1) # Shape: [1, 257, dino_dim]
        else:
            tokens = batch

        # Pass through visual adapter/projection without final pooling
        # Looking for common adapter attributes in custom CLIP visual towers
        visual_encoder = model.visual
        if hasattr(visual_encoder, "adapter"):
            # If your model has an explicit visual adapter module
            projected_tokens = visual_encoder.adapter(tokens)
        elif hasattr(visual_encoder, "proj") and callable(visual_encoder.proj):
            projected_tokens = visual_encoder.proj(tokens)
        else:
            # Fallback: if model.visual has submodules or we can pass through trunk/layers
            # Let's try passing tokens through visual_encoder if it accepts sequence inputs
            try:
                projected_tokens = visual_encoder(tokens) if visual_encoder(tokens).ndim == 3 else None
            except Exception:
                projected_tokens = None

        # If projection still returned pooled output, manually apply the adapter if accessible
        if projected_tokens is None or projected_tokens.ndim != 3:
            # Inspecting common adapter names or fallback to linear projection if available
            for name, module in visual_encoder.named_modules():
                if "adapter" in name.lower() or "proj" in name.lower():
                    try:
                        test_out = module(tokens)
                        if test_out.ndim == 3:
                            projected_tokens = test_out
                            break
                    except Exception:
                        continue

        if projected_tokens is None or projected_tokens.ndim != 3:
            raise ValueError(
                "Could not extract 3D token sequence from `model.visual`. "
                "Ensure your visual model exposes unpooled token outputs."
            )

        # Drop the CLS token at index 0, leaving 256 patch tokens: [1, 256, 256]
        patch_embeddings = projected_tokens[:, 1:, :]
        patch_embeddings = F.normalize(patch_embeddings, dim=-1)

        # 2. Encode 5 Text Queries
        enc = tokenizer(
            queries,
            padding=True,
            truncation=True,
            max_length=model.transformer.context_length,
            return_tensors="pt",
        )
        tokens_dict = {k: v.to(device) for k, v in enc.items() if k in ("input_ids", "attention_mask")}
        text_embeddings = model.encode_text(tokens_dict) # [5, embed_dim]
        text_embeddings = F.normalize(text_embeddings, dim=-1) # [5, embed_dim]

        # 3. Compute Cosine Similarity: [5, 256]
        similarities = torch.matmul(text_embeddings, patch_embeddings[0].T)

    # 4. Reshape and Upsample Heatmaps
    fig, axes = plt.subplots(1, 6, figsize=(20, 4))
    axes[0].imshow(pil_resized)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    for idx, query in enumerate(queries):
        sim_vector = similarities[idx] # [256]
        grid_size = int(np.sqrt(sim_vector.shape[0])) # 16
        heatmap = sim_vector.reshape(grid_size, grid_size).cpu().float()

        # Upsample to 224x224 bilinear
        heatmap_resized = F.interpolate(
            heatmap.unsqueeze(0).unsqueeze(0), 
            size=(224, 224), 
            mode='bilinear', 
            align_corners=False
        ).squeeze().numpy()

        # Normalize 0 to 1
        h_min, h_max = heatmap_resized.min(), heatmap_resized.max()
        if h_max > h_min:
            heatmap_normalized = (heatmap_resized - h_min) / (h_max - h_min)
        else:
            heatmap_normalized = heatmap_resized

        axes[idx + 1].imshow(pil_resized)
        axes[idx + 1].imshow(heatmap_normalized, cmap='jet', alpha=0.25)
        axes[idx + 1].set_title(f"Q: {query}", fontsize=10)
        axes[idx + 1].axis("off")

    plt.tight_layout()
    plt.show()

# ==========================================
# Example Usage
# ==========================================
PARQUET_PATH = "/data/susanket/vlm/images3/image-07.parquet"
df = pd.read_parquet(PARQUET_PATH)

# Pick a single random row (or specific index like df.iloc[0])
sample_row = df.iloc[2]

# Supply exactly 5 phrases
MY_5_PHRASES = [
    "road",
    "trees",
    "trees",
    "black roof",
    "swimming pool"
]

# Run the visualization
visualize_phrase_heatmaps(sample_row, MY_5_PHRASES, model, tokenizer, dino_backbone)

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np

def plot_image_collage(df, start_idx, end_idx, image_column="image_bytes", id_column=None, max_cols=5):
    """
    Plots a collage of images from row index start_idx to end_idx from a DataFrame.
    
    Args:
        df: Pandas DataFrame containing the images.
        start_idx: Starting row index (inclusive).
        end_idx: Ending row index (exclusive, like iloc slicing).
        image_column: Column name containing the raw image bytes or dict.
        id_column: Optional column name for a specific ID/Name. If None, uses DataFrame index.
        max_cols: Maximum number of images per row in the collage grid.
    """
    subset = df.iloc[start_idx:end_idx]
    num_images = len(subset)
    
    if num_images == 0:
        print("No rows found in the specified range.")
        return

    # Calculate grid dimensions
    cols = min(num_images, max_cols)
    rows = math.ceil(num_images / cols)
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    
    # Handle single-row case where axes is not a multi-dimensional array
    if rows == 1 and cols == 1:
        axes = np.array([axes])
    else:
        axes = np.array(axes).flatten()

    for i, (df_idx, row) in enumerate(subset.iterrows()):
        # Decode image using your existing helper function
        img = decode_image(row[image_column])
        
        # Get the label (either from a specified column or the dataframe index)
        row_label = row[id_column] if (id_column and id_column in row) else df_idx
        
        axes[i].imshow(img)
        axes[i].set_title(f"ID: {row_label}", fontsize=9)
        axes[i].axis("off")

    # Hide any remaining unused subplots in the grid
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

# ==========================================
# Example Usage:
# ==========================================
# Plot rows 0 through 10 (10 images total), 5 images per row
plot_image_collage(df, start_idx=100, end_idx=1000, image_column="image_bytes", max_cols=5)

In [ ]:
df

### Cone Viz

In [ ]:
text = pd.read_parquet(rf"/data/susanket/vlm/images3/caption-00.parquet").set_index('file_name').loc[df['file_name']].reset_index()

QUERIES = text.summary.to_list()

with torch.no_grad():
    enc = tokenizer(
        QUERIES,
        padding=True,
        truncation=True,
        max_length=model.transformer.context_length,
        return_tensors="pt",
    )
    tokens = {k: v.to(device) for k, v in enc.items() if k in ("input_ids", "attention_mask")}
    
    text_embeddings = model.encode_text(tokens)
    text_embeddings = F.normalize(text_embeddings, dim=-1).cpu().numpy()

In [ ]:
import plotly.graph_objects as go
import numpy as np
from sklearn.decomposition import PCA

# 1. Assume you have:
# - image_embeddings: np.ndarray of shape (N, D) - already L2 normalized
# - text_embeddings: np.ndarray of shape (N, D) - already L2 normalized (paired with images)

# If your embedding dimension D > 3, reduce them to 3D using PCA
# (Note: we re-normalize to project the 3D points back onto the unit sphere surface)
use_pca = image_embeddings.shape[1] > 3
if use_pca:
    print("Reducing embeddings to 3D using PCA for visualization...")
    pca = PCA(n_components=3)
    # Fit PCA on combined embeddings to share the same 3D subspace
    combined = np.concatenate([image_embeddings, text_embeddings], axis=0)
    pca.fit(combined)
    
    img_3d = pca.transform(image_embeddings)
    txt_3d = pca.transform(text_embeddings)
    
    # Re-normalize to project back onto the unit sphere surface
    img_3d = img_3d / np.linalg.norm(img_3d, axis=1, keepdims=True)
    txt_3d = txt_3d / np.linalg.norm(txt_3d, axis=1, keepdims=True)
else:
    img_3d = image_embeddings[:, :3]
    txt_3d = text_embeddings[:, :3]

# 2. Create a wireframe/surface for the translucent unit sphere
phi = np.linspace(0, np.pi, 30)
theta = np.linspace(0, 2 * np.pi, 30)
phi, theta = np.meshgrid(phi, theta)

xs = np.sin(phi) * np.cos(theta)
ys = np.sin(phi) * np.sin(theta)
zs = np.cos(phi)

sphere_trace = go.Surface(
    x=xs, y=ys, z=zs,
    colorscale=[[0, 'rgba(200,220,240,0.15)'], [1, 'rgba(200,220,240,0.15)']],
    showscale=False,
    hoverinfo='none'
)

# 3. Create lines connecting paired Image <-> Text embeddings
line_x, line_y, line_z = [], [], []
# Let's plot lines for a subset (e.g., first 100 samples) to avoid cluttering the plot
num_lines_to_plot = min(100, len(img_3d)) 

for i in range(num_lines_to_plot):
    line_x.extend([img_3d[i, 0], txt_3d[i, 0], None])
    line_y.extend([img_3d[i, 1], txt_3d[i, 1], None])
    line_z.extend([img_3d[i, 2], txt_3d[i, 2], None])

lines_trace = go.Scatter3d(
    x=line_x, y=line_y, z=line_z,
    mode='lines',
    line=dict(color='rgba(100, 100, 100, 0.3)', width=1),
    hoverinfo='none',
    name='Pairs'
)

# 4. Image embeddings scatter (Blue)
img_trace = go.Scatter3d(
    x=img_3d[:num_lines_to_plot, 0],
    y=img_3d[:num_lines_to_plot, 1],
    z=img_3d[:num_lines_to_plot, 2],
    mode='markers',
    marker=dict(size=4, color='blue', opacity=0.8),
    name='Image Embeddings'
)

# 5. Text embeddings scatter (Red)
txt_trace = go.Scatter3d(
    x=txt_3d[:num_lines_to_plot, 0],
    y=txt_3d[:num_lines_to_plot, 1],
    z=txt_3d[:num_lines_to_plot, 2],
    mode='markers',
    marker=dict(size=4, color='red', opacity=0.8),
    name='Text Embeddings'
)

# 6. Assemble layout and render figure
fig = go.Figure(data=[sphere_trace, lines_trace, img_trace, txt_trace])

fig.update_layout(
    title="CLIP Embedding Space on Unit Sphere",
    scene=dict(
        xaxis=dict(showbackground=False, visible=False),
        yaxis=dict(showbackground=False, visible=False),
        zaxis=dict(showbackground=False, visible=False),
        aspectmode='cube'
    ),
    margin=dict(r=0, l=0, b=0, t=40)
)

# Replace fig.show() with this:
fig.write_html("embedding_sphere.html")
print("Saved interactive plot to embedding_sphere.html")

In [ ]:
import os
import io
import sys
import tqdm
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
import open_clip  # Make sure you pip install open_clip_torch
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.decomposition import PCA

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 1. LOAD GEORSCLIP MODEL & TOKENIZER
# ==========================================
# Choose your architecture variant, e.g., "ViT-B-32" or "ViT-H-14"
MODEL_ARCH = "ViT-B-32" 
CKPT_PATH = "/data/susanket/vlm/sota/ckpt/RS5M_ViT-B-32.pt"  # Update to your downloaded GeoRSCLIP checkpoint path
# from huggingface_hub import hf_hub_download

# # This will correctly fetch the binary file from Hugging Face LFS
# CKPT_PATH = hf_hub_download(repo_id="Zilun/GeoRSCLIP", filename="RS5M_ViT-B-32.pt")

# Create model and corresponding default preprocessing transforms from open_clip
model, _, image_transform = open_clip.create_model_and_transforms(MODEL_ARCH, pretrained=None)

# Load GeoRSCLIP state dict weights
checkpoint = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
# Handle potential wrapper keys if weights are saved under "state_dict" or "model"
state_dict = checkpoint["state_dict"] if "state_dict" in checkpoint else checkpoint
state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()} # clean DDP prefix if present

model.load_state_dict(state_dict, strict=False)
model = model.to(device)
model.eval()

# Get the OpenCLIP tokenizer
tokenizer = open_clip.get_tokenizer(MODEL_ARCH)
print("GeoRSCLIP successfully loaded!")

# ==========================================
# 2. LOAD DATA & PROCESS IMAGES (Using Parquet)
# ==========================================
PARQUET_PATH = "/data/susanket/vlm/images3/image-00.parquet"  # Update path
IMAGE_COLUMN = "image_bytes"  # Column name containing raw image bytes

df = pd.read_parquet(PARQUET_PATH).sample(10_000)
print(f"Loaded {len(df)} samples from {PARQUET_PATH}")

def decode_and_transform(raw_image_data):
    if isinstance(raw_image_data, dict) and "bytes" in raw_image_data:
        raw_bytes = raw_image_data["bytes"]
    else:
        raw_bytes = raw_image_data
    try:
        img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
    except Exception:
        img = Image.new("RGB", (224, 224))
    return img

raw_pil_images = [decode_and_transform(row[IMAGE_COLUMN]) for _, row in tqdm.tqdm(df.iterrows())]

# Apply OpenCLIP's native transform pipeline
tensor_images = torch.stack([image_transform(img) for img in tqdm.tqdm(raw_pil_images)])

# ==========================================
# 3. EXTRACT IMAGE EMBEDDINGS
# ==========================================
BATCH_SIZE = 256
all_img_embeddings = []

with torch.no_grad():
    for i in range(0, len(tensor_images), BATCH_SIZE):
        batch = tensor_images[i : i + BATCH_SIZE].to(device)
        embeddings = model.encode_image(batch)
        embeddings = F.normalize(embeddings, dim=-1)
        all_img_embeddings.append(embeddings.cpu().numpy())

image_embeddings = np.concatenate(all_img_embeddings, axis=0)
print(f"Computed image embeddings matrix shape: {image_embeddings.shape}")

# ==========================================
# 4. EXTRACT TEXT EMBEDDINGS (Captions from df or queries)
# ==========================================
# Example: Using text queries or extracting captions from a text column in df
text = pd.read_parquet(rf"/data/susanket/vlm/images3/caption-00.parquet").set_index('file_name').loc[df['file_name']].reset_index()

QUERIES = text.summary.to_list()

with torch.no_grad():
    text_tokens = tokenizer(QUERIES).to(device)
    text_embeddings = model.encode_text(text_tokens)
    text_embeddings = F.normalize(text_embeddings, dim=-1).cpu().numpy()


# ==========================================
# 5. 3D UNIT SPHERE VISUALIZATION (Plotly)
# ==========================================
# If your text and image embeddings are paired 1-to-1 (e.g., image features vs caption features from parquet),
# you can feed them directly here. For queries, ensure shapes align or map them appropriately.
use_pca = image_embeddings.shape[1] > 3
if use_pca:
    print("Reducing embeddings to 3D using PCA for visualization...")
    pca = PCA(n_components=3)
    combined = np.concatenate([image_embeddings[:100], text_embeddings[:100]], axis=0)
    pca.fit(combined)
    
    img_3d = pca.transform(image_embeddings[:100])
    txt_3d = pca.transform(text_embeddings[:100])
    
    # Re-normalize to project back onto the unit sphere surface
    img_3d = img_3d / np.linalg.norm(img_3d, axis=1, keepdims=True)
    txt_3d = txt_3d / np.linalg.norm(txt_3d, axis=1, keepdims=True)
else:
    img_3d = image_embeddings[:100, :3]
    txt_3d = text_embeddings[:100, :3]

# Sphere mesh grid
phi = np.linspace(0, np.pi, 30)
theta = np.linspace(0, 2 * np.pi, 30)
phi, theta = np.meshgrid(phi, theta)
xs, ys, zs = np.sin(phi) * np.cos(theta), np.sin(phi) * np.sin(theta), np.cos(phi)

sphere_trace = go.Surface(
    x=xs, y=ys, z=zs,
    colorscale=[[0, 'rgba(200,220,240,0.15)'], [1, 'rgba(200,220,240,0.15)']],
    showscale=False, hoverinfo='none'
)

line_x, line_y, line_z = [], [], []
for i in range(len(img_3d)):
    line_x.extend([img_3d[i, 0], txt_3d[i, 0], None])
    line_y.extend([img_3d[i, 1], txt_3d[i, 1], None])
    line_z.extend([img_3d[i, 2], txt_3d[i, 2], None])

lines_trace = go.Scatter3d(x=line_x, y=line_y, z=line_z, mode='lines', line=dict(color='rgba(100, 100, 100, 0.3)', width=1), hoverinfo='none')
img_trace = go.Scatter3d(x=img_3d[:, 0], y=img_3d[:, 1], z=img_3d[:, 2], mode='markers', marker=dict(size=4, color='blue', opacity=0.8), name='Images')
txt_trace = go.Scatter3d(x=txt_3d[:, 0], y=txt_3d[:, 1], z=txt_3d[:, 2], mode='markers', marker=dict(size=4, color='red', opacity=0.8), name='Texts')

fig = go.Figure(data=[sphere_trace, lines_trace, img_trace, txt_trace])
fig.update_layout(
    title="GeoRSCLIP Embedding Space on Unit Sphere",
    scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False), aspectmode='cube'),
    margin=dict(r=0, l=0, b=0, t=40)
)

# If running in a script without nbformat issues, or save to HTML:
fig.write_html("georsclip_sphere.html")
print("Saved interactive plot to georsclip_sphere.html")